# 07 — Discovering STAC catalogs from browser-native notebooks

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/07_stac_browser_catalogs.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/07_stac_browser_catalogs.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `b262de9f1ddc457606bba1124a084a709de5356f91d776c75672a864083f2d68`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 07 — Discovering STAC catalogs from browser-native notebooks

This lab connects four public STAC Browsers to their machine-readable catalog APIs: [UN Biodiversity Lab](https://stac.unbiodiversitylab.org/?.language=en), [Copernicus Data Space](https://browser.stac.dataspace.copernicus.eu/), [USGS LandsatLook](https://landsatlook.usgs.gov/stac-browser/?.language=en), and [WorldPop](https://stac.worldpop.org/?.language=en).

A STAC Browser is a human-friendly client. A catalog can be static or offer a searchable API; follow its advertised capabilities.

**Read first:** [What is a STAC Browser?](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/platforms/stac-discovery.html) connects the companion Explorer to a metadata checklist, an offline Python exercise and this lab's browser-access boundaries.

## Start here: the question behind the code

**How do we find an Earth dataset and understand its access path?**

**What you will learn:** browse → follow → inspect → check access. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Discovery foundation: STAC organizes metadata; this notebook does not train a model.

**Why this is useful:** Reliable discovery records location, time, links and licensing before a GeoAI workflow spends resources on image assets.

**Where the work happens:** The notebook offers browser links and a schematic graph. Direct API probing is intentionally skipped in JupyterLite to avoid unreliable provider requests.

![Read the catalog before the pixels. Keep human-facing browser links beside machine-facing API addresses. Catalog and collection links lead toward item metadata. An item describes geometry and time and links to data or thumbnails. A working catalog page does not prove every data asset accepts browser requests.](assets/learning/07/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: Read the catalog before the pixels — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/07/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/07/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/07/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=07_stac_browser_catalogs.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/07_stac_browser_catalogs.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/07/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Browse (0–8 s):** Keep human-facing browser links beside machine-facing API addresses. Code landmark: `CATALOGS`.

2. **Follow (8–16 s):** Catalog and collection links lead toward item metadata. Code landmark: `root.get('links', [])`.

3. **Inspect (16–24 s):** An item describes geometry and time and links to data or thumbnails. Code landmark: `Item → assets`.

4. **Check access (24–32 s):** A working catalog page does not prove every data asset accepts browser requests. Code landmark: `browser_runtime`.


## See the connections

![STAC: discover before downloading: Catalog: Links connect collections and items; Item: Geometry, time and assets describe a scene; Access: Metadata access differs from asset access; Check: Browser CORS, license and media type](assets/07_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == 'emscripten':
    import micropip
    await micropip.install(['geolibre==3.0.0', 'pyodide-http'])
    import pyodide_http
    pyodide_http.patch_all()
from geolibre_lite import LiteMap as Map
import requests
browser_runtime = sys.platform == 'emscripten'
from IPython.display import display, Markdown


### Code guide 2 — Keep provider addresses separate from requests

**Read the Python:** Each catalog has a human-facing browser URL and a machine-facing API URL. `get_json` wraps request, HTTP checking and JSON parsing. The loop deliberately skips direct probes in JupyterLite; in full Python it records successes or exceptions per provider so one failure does not stop the table.

**Look for:** Four status rows. A skipped browser query is a policy choice in this notebook, not proof the provider is currently down.

**Pause and try:** Locate `continue` and explain why no API request happens after it in the browser branch.


In [ ]:
import sys
import requests
browser_runtime = sys.platform == 'emscripten'
CATALOGS = {
    'UN Biodiversity Lab': {
        'api': 'https://unbl-prod-stac.azurewebsites.net/',
        'browser': 'https://stac.unbiodiversitylab.org/?.language=en',
    },
    'Copernicus Data Space': {
        'api': 'https://stac.dataspace.copernicus.eu/v1/',
        'browser': 'https://browser.stac.dataspace.copernicus.eu/',
    },
    'USGS LandsatLook': {
        'api': 'https://landsatlook.usgs.gov/stac-server/',
        'browser': 'https://landsatlook.usgs.gov/stac-browser/?.language=en',
    },
    'WorldPop': {
        'api': 'https://api.stac.worldpop.org',
        'browser': 'https://stac.worldpop.org/?.language=en',
    },
}

def get_json(url, **kwargs):
    response = requests.get(url, timeout=60, **kwargs)
    response.raise_for_status()
    return response.json()

catalog_rows = []
catalog_roots = {}
for name, info in CATALOGS.items():
    if browser_runtime:
        catalog_rows.append({
            'catalog': name,
            'title': 'Use the linked STAC Browser',
            'stac_version': None,
            'api_root': info['api'],
            'browser': info['browser'],
            'status': 'Direct API query skipped in JupyterLite to avoid provider CORS restrictions',
        })
        continue
    try:
        root = get_json(info['api'].rstrip('/') + '/')
        catalog_roots[name] = root
        catalog_rows.append({
            'catalog': name,
            'title': root.get('title'),
            'stac_version': root.get('stac_version'),
            'api_root': info['api'],
            'browser': info['browser'],
            'status': 'API reachable from this browser',
        })
    except Exception as exc:
        # A STAC Browser can work even when its API blocks cross-origin XHR.
        catalog_rows.append({
            'catalog': name,
            'title': 'Unavailable to in-browser requests',
            'stac_version': None,
            'api_root': info['api'],
            'browser': info['browser'],
            'status': f'Browser CORS/authentication restriction: {type(exc).__name__}',
        })

import pandas as pd
pd.DataFrame(catalog_rows)


## Open the human-facing browsers

Use these links to inspect the same catalogs visually, then use the API URLs above for reproducible code. JupyterLite skips direct API requests because catalog CORS policies vary; in a normal CPython notebook, the API metadata query runs directly.

### Code guide 3 — Make readable links from a dictionary

**Read the Python:** An f-string fills each provider's name and URL into Markdown link syntax. The generator visits all providers and `'\n'.join(...)` combines their lines. `Markdown` tells Jupyter how to render the resulting text.

**Look for:** Clickable links to four STAC browsers alongside their API roots.

**Pause and try:** Open one browser and identify a collection title and its license.


In [ ]:
display(Markdown('\n'.join(f"- [{name}]({info['browser']}) — API: `{info['api']}`" for name, info in CATALOGS.items())))


### Code guide 4 — Follow the catalog's advertised relationships

**Read the Python:** `dict.get` supplies a default if a field is absent. A set removes duplicate relationship names, and `sorted` makes the result readable. Subtracting the available-root names from all catalog names finds those that were not inspected.

**Look for:** Relationship names in full Python when queries succeeded; a skipped list is expected in JupyterLite.

**Pause and try:** Treat the final 'blocked' message as 'not inspected' for a deliberately skipped query; it is not a measured CORS diagnosis.


In [ ]:
# The catalog's links advertise the standard endpoints a client can follow.
for name, root in catalog_roots.items():
    rels = sorted({link.get('rel') for link in root.get('links', []) if link.get('rel')})
    print(name, '=>', ', '.join(rels))
missing = sorted(set(CATALOGS) - set(catalog_roots))
if missing:
    print('Skipped from direct API introspection because the browser blocked the request:', ', '.join(missing))


## GeoLibre context map

The map is not downloading imagery here. It gives the four catalogs a geographic context; the next notebook turns STAC search results into GeoAI-ready features.

### Code guide 5 — Use a context map without inventing coverage

**Read the Python:** Each marker includes a provider link in its properties. These hand-placed points are illustrative context markers; they are neither provider headquarters nor the boundaries of catalog coverage.

**Look for:** Four clickable context markers. No imagery search or download occurs in this cell.

**Pause and try:** Explain why a point at 0,0 cannot describe the geographic extent of UN Biodiversity Lab.


In [ ]:
m = Map(center=(-20, 20), zoom=1.5, height='560px')
m.add_marker(-74.0, 40.7, name='USGS / LandsatLook', properties={'browser': CATALOGS['USGS LandsatLook']['browser']})
m.add_marker(12.5, 42.0, name='Copernicus Data Space', properties={'browser': CATALOGS['Copernicus Data Space']['browser']})
m.add_marker(0, 0, name='UN Biodiversity Lab', properties={'browser': CATALOGS['UN Biodiversity Lab']['browser']})
m.add_marker(104.9, 12.6, name='WorldPop', properties={'browser': CATALOGS['WorldPop']['browser']})
m


## Data & software citations

- [STAC specification](https://stacspec.org/) and [STAC API specification](https://github.com/radiantearth/stac-api-spec).
- [UN Biodiversity Lab STAC Browser](https://stac.unbiodiversitylab.org/?.language=en); public API root: `https://unbl-prod-stac.azurewebsites.net/`.
- [Copernicus Data Space STAC Browser](https://browser.stac.dataspace.copernicus.eu/); public API root: `https://stac.dataspace.copernicus.eu/v1/`.
- [USGS LandsatLook STAC Browser](https://landsatlook.usgs.gov/stac-browser/?.language=en); public API root: `https://landsatlook.usgs.gov/stac-server/`.
- [WorldPop STAC Browser](https://stac.worldpop.org/?.language=en); public API root: `https://api.stac.worldpop.org`.
- [GeoLibre](https://geolibre.app/) and [GeoAI](https://opengeoai.org/).

### Code guide 6 — Load the plotting tools

**Read the Python:** `from IPython.display import display` lets a cell show objects explicitly. In Pyodide, `await loadPackage('matplotlib')` loads a compiled plotting package. The `if` block is skipped in ordinary Python, where the package must already be installed.

**Look for:** This preparation cell may show no figure; the next cell creates it.

**Pause and try:** Keep this cell before the chart cell when you rerun the notebook after a kernel restart.


In [ ]:
from IPython.display import display
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage("matplotlib")

## Make the catalog graph explicit

Read this relationship graph from left to right. An item can have several assets with different roles. A footprint says where an item is located; it does not guarantee every pixel is valid or accessible.

### Code guide 7 — Read STAC as a graph of links

**Read the Python:** `nodes` gives diagram labels plot coordinates, not Earth coordinates. `edges` lists the arrows. The first loop draws relationships and the second draws labeled boxes. `axis('off')` hides meaningless spatial axes.

**Look for:** A schematic catalog → collection → item → asset graph.

**Pause and try:** Trace a path to a thumbnail and a different path to a COG; say which one contains analysis pixels.


In [ ]:
from cognition import style_plots
import matplotlib.pyplot as plt
style_plots()
fig, ax = plt.subplots(figsize=(11, 4))
nodes = {"Catalog": (0,1), "Collection": (1,1), "Item A": (2,1.7), "Item B": (2,.3),
         "COG bands": (3,2.1), "Thumbnail": (3,1.2), "Metadata": (3,.3)}
edges = [("Catalog","Collection"),("Collection","Item A"),("Collection","Item B"),
         ("Item A","COG bands"),("Item A","Thumbnail"),("Item B","Metadata")]
for a,b in edges:
    ax.annotate("",nodes[b],nodes[a],arrowprops={"arrowstyle":"->","color":"#64748b","shrinkA":35,"shrinkB":35})
for label,(x,y) in nodes.items():
    ax.text(x,y,label,ha="center",va="center",bbox={"boxstyle":"round,pad=.7","fc":"#dbeafe","ec":"#2563eb"})
ax.set(xlim=(-.5,3.6), ylim=(-.3,2.6), title="STAC relationship graph (schematic, not a queried catalog)")
ax.axis("off")
plt.show()

**Trace it:** choose one linked browser above and identify a collection, item, thumbnail and data asset. Record the actual license and access policy; a visible thumbnail does not prove that the full asset is public.

## Make sense of the result

![I found a thumbnail. Do I have all the data? Check the item, asset type, license and access rules. Discovery comes before downloading and learning.](assets/learning/07/cartoon.svg)

**Cartoon transcript:** I found a thumbnail. Do I have all the data? → Check the item, asset type, license and access rules. → Discovery comes before downloading and learning.

**Explain it to a colleague:** Why does a skipped request differ from an empty catalog?

<details><summary>Check your reasoning</summary>

A skipped request collected no evidence about contents. An empty response only describes that particular query.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
